### **Student Information**
GitHub ID: brendatan99

Kaggle name: brendatt

Kaggle private scoreboard snapshot: 

![pic_ranking.png](./pics/pic_ranking.png)

*Your name and student ID are deliberately not asked for here: this notebook goes into your repository, which is public. They belong in the git-ignored `config/.env`, where the other two notebooks read them from, and your NTU COOL submission is what ties the repository link back to you.*

---

# **Instructions**

This lab has **three parts**: exercises in the Master notebook, exercises and an agent conversation in the AgentPipeline notebook, and an internal Kaggle competition documented in this Homework notebook. Together they add up to **100 points**.

**Environment recommendations to solve lab 2:**
- **Part 1 and 2 exercises:** Everything can be run with only CPU.
- **Part 3 exercises:** For the competition you will probably need GPU to train your models, so it is recommended to use Colab or Kaggle if you don't have a laptop with a dedicated GPU.

## **Part 1 (40 pts):**

1. __Main Exercises (35 pts):__ Do the **take home exercises** from Sections: `1. Data Preparation` to `8. Information Retrieval and RAG`, in the DM2026-Lab2-Master Notebook. Total: `13 exercises`. Commit your code and submit the repository link to NTU Cool.

2. **Code Comments (5 pts):** **Tidy up the code in your notebook**. 

## **Part 2 (30 pts):**

1. **Agent Prompt (20 pts):** Interact with the Agent using appropriate prompts to guide it through and complete all the required tasks.

2. **Main Exercises (10 pts):** Do the remaining **take home exercises** from DM2026-Lab2-AgentPipeline Notebook. Total: `3 exercises required`. Commit your code and submit the repository link to NTU Cool.

## **Part 3 (30 pts):**

1. **Kaggle Competition Participation (15 pts):** Submit your predictions to the [Data Mining Class Kaggle Competition](https://www.kaggle.com/t/d99def2e62194b0d8f633267c32e6d1e) by **November 22**. Check Kaggle for the exact closing time. The competition concerns emotion recognition on Twitter; participation points use the Private Leaderboard ranking:
    - If your submission meets or exceeds the **baseline score (F1 score: 0.5)**, and there are no    significant issues with your report or code, you will automatically receive a base score of 5 points of the 15 points.

        The remaining 10 points will be determined by your ranking among participants who meet or exceed the baseline:

        |Ranking  |Additional Points  |
        | --- | --- |
        |Top 10% |+10 |
        |Top 20% |+9 |
        |Top 30% |+8 |
        |Top 40% |+7 |
        |Top 50% |+6 |
        |Top 60% |+5 |
        |Top 70% |+4 |
        |Top 80% |+3 |
        |Top 90% |+2 |
        |<Top 90% |+1 |

    - If your submission does not reach the baseline score, your grade will be lower than 5. The score will range from 0 to 4 points, depending on your Private Leaderboard ranking and the relative performance of students whose submissions are also below the baseline.

2. **Competition Report (15 pts)** A report section to be filled in inside this notebook in Markdown Format, we already provided you with the template below. You need to describe your work developing the model for the competition. The report should include a section describing briefly the following elements: 
* All of your AI prompts and what you decide do use.
* Your preprocessing steps.
* The feature engineering steps.
* Explanation of your model.

[Markdown Guide - Basic Syntax](https://www.markdownguide.org/basic-syntax/)

**`Things to note for Part 3:`**

* **The code used for the competition should be in this Jupyter Notebook File** `DM2026-Lab2-Homework.ipynb`.

* **Push the code used for the competition to your repository**.

* **The code should have a clear separation for the same sections of the report, AI prompts, preprocessing, feature engineering and model explanation. Briefly comment your code for easier understanding, we provide a template at the end of this notebook.**

* Save a screenshot of your Kaggle Private Leaderboard ranking as `pics/pic_ranking.png` and keep the image link in the Student Information cell at the top of this notebook.

The Kaggle competition closes on **November 22**. The three days afterward are for completing this notebook's Markdown report, checking your files, and submitting the lab by **November 25 at 11:59 pm (Wednesday)**.

Push your work to your repository, then submit its link and a ZIP on the corresponding NTU COOL assignment. The ZIP must contain `DM2026-Lab2-Master.ipynb`, `DM2026-Lab2-AgentPipeline.ipynb`, `DM2026-Lab2-Homework.ipynb`, both encrypted logs in `session_logs/`, and `pics/pic_ranking.png`. Name it with your student ID and your name. The two logs are git-ignored, so they are not in your repository and the ZIP is the only copy that reaches us: open it before uploading and check both are inside. If you miss them, you can upload them on NTU COOL until **November 26 at 11:59 pm**. See `README.md` for the exact file paths and setup steps.

---

Next you will find the template report with some simple markdown syntax explanations, use it to structure your content.

You can delete the syntax suggestions after you use them.

---

***

# **Project Report**

---

## 1. Model Development

### 1.1 Preprocessing Steps

The competition dataset provides the text and labels across three separate files: `final_posts.json` contains 4,800 posts, `data_identification.csv` divides them into 3,360 training and 1,440 test instances, and `emotion.csv` supplies the training labels. I merged these files on `post_id` and then checked the result with assertions rather than assuming it worked: all 4,800 rows match across files, `post_id` is unique, and no text entries are missing. A row quietly dropped here would have changed every number in the experiments that follow.

Two specific details identified during this verification shaped the subsequent pipeline. First, the target column in `emotion.csv` is named `label` rather than `emotion`, and consists of six exact strings: `anger`, `disgust`, `fear`, `joy`, `sadness`, and `surprise`. The submission script validates predictions against this exact set to prevent any formatting or spelling discrepancies in the final output. Second, the test instances in the merged dataframe do not follow the row order of `sampleSubmission.csv`. Generating predictions in the default dataframe order and exporting them directly would misalign almost every label, even though the resulting file might appear valid at first glance. Consequently, every prediction set is explicitly joined onto `sampleSubmission.csv` by ID prior to export.

Exploratory analysis of the training data reveals notable class imbalance and text length skewness. The six classes are unevenly distributed, ranging from joy at 26.04% down to fear at 8.57% (a ratio of 3.04). Because the official evaluation metric is macro F1, which weights rare and common classes equally, standard accuracy is an inadequate proxy. Furthermore, text length is heavily right-skewed, with a median of 13 words and a maximum of 508 words. To prevent exceptionally long posts from dominating the feature vectors solely through term repetition, sublinear term frequency scaling (`sublinear_tf=True`) is applied across all TF-IDF representations.

Constructing a leak-free validation split is the most critical step in the preprocessing pipeline. Prior to model training, the text is normalized by converting characters to lowercase and collapsing whitespace to identify duplicate entries. Within the training set, 50 distinct texts appear multiple times, accounting for 100 rows in total, and 16 of these duplicate groups contain contradictory labels. Applying a standard `StratifiedKFold` would distribute identical posts across both training and validation splits, inflating validation scores through data memorization. To prevent this leakage, the dataset is partitioned using `StratifiedGroupKFold`, grouping by the MD5 hash of the normalized text. Verification confirms that zero groups span multiple folds, while class distributions remain within 0.3 percentage points across all five splits. This partition is saved to `folds.csv` and kept fixed across all experiments, ensuring that performance metrics from linear baselines to fine-tuned transformers remain directly comparable.

Two additional dataset characteristics were documented and intentionally left unmodified. First, the 16 duplicate groups with contradictory labels establish an upper bound on achievable performance. I kept these instances in the training set, because removing them would artificially inflate the validation score without actually making the model generalize better. Second, 59 distinct texts appear in both the training and test sets, meaning the labels for 4.1% of the test instances are effectively known. Evaluating the three-component ensemble on these 59 overlapping instances showed that the model already predicts 55 of them correctly. Overriding these predictions by hand would change only 4 out of 1,440 rows while adding an ad-hoc rule to the pipeline, so I decided not to do it.

### 1.2 Feature Engineering Steps

I built three distinct feature sets and evaluated them on the fixed validation folds, each designed to capture a different linguistic level of the tweets.

**Word-level features.** A word-level TF-IDF vectorizer over unigrams and bigrams (`min_df=2`, `sublinear_tf=True`) directly captures explicit emotional vocabulary, allowing the classifier to assign weights to specific terms such as *furious*, *terrified*, and *blessed*. Evaluated independently, this representation achieves an out-of-fold macro F1 score of 0.4730.

**Character-level features.** Because social media text frequently contains informal spelling and deliberate word lengthening, a word-level tokenizer treats variations like *angryyy* and *angry* as entirely separate tokens. Preliminary analysis in the Master notebook demonstrated that sub-word FastText embeddings assign a cosine similarity of 0.984 to *angryyy* and *angry*, compared to 0.624 for a character transposition like *anrgy*. This confirms that sub-word representations effectively capture character repetition, even if they do not resolve all typographical errors. Using a `char_wb` TF-IDF vectorizer over 2-to-5 character n-grams achieves an out-of-fold macro F1 of 0.5045 on its own, outperforming the word-level features. Concatenating the word and character representations increases the score to 0.5146.

**Dense semantic embeddings.** Sparse n-gram representations cannot capture semantic similarity between distinct words, such as *afraid* and *scared*. In earlier experiments, self-trained Word2Vec embeddings yielded a similarity of only 0.482 for this pair, whereas pre-trained MiniLM embeddings reached 0.845. Incorporating L2-normalized, 384-dimensional MiniLM sentence embeddings alongside the two sparse TF-IDF matrices raises the out-of-fold macro F1 from 0.5146 to 0.5708. This 0.0562 improvement corresponds to roughly 81 test instances and far exceeds the gains from classifier hyperparameter tuning. The resulting combined feature matrix consists of 35,420 columns.

**End-to-end representation learning.** While combining TF-IDF and static embeddings relies on predefined feature extraction, fine-tuning a pre-trained transformer adapts the text representation and the classification head jointly. Given that 3,360 training instances are insufficient to learn language representations from scratch, the pipeline utilizes `cardiffnlp/twitter-roberta-base-2022-154m`, a checkpoint pre-trained on 154 million tweets. Input text is normalized according to the checkpoint's pre-training convention, masking user handles as `@user` and URLs as `http`.

**Starting from a model that already knows emotions.** The tweet-domain checkpoint knows how people write on Twitter, but nobody ever told it what an emotion is. A second family of checkpoints has been told. `j-hartmann/emotion-english-distilroberta-base` and its RoBERTa-large sibling were trained on six English emotion datasets with seven classes, our six plus *neutral*. Loading them the usual way throws that classification head away and starts a new one from random weights. I kept the head instead and sliced out the six rows that match our labels, looked up by name in `config.id2label`, so fine-tuning starts from a classifier that can already tell *anger* from *disgust* to some degree. Three more emotion checkpoints were loaded the same way (Section 3.6), and the text normalisation is the same as for the tweet model.

A notable difference between the sparse and transformer representations lies in how they process emojis. The default regular expression in `TfidfVectorizer`, `(?u)\b\w\w+\b`, retains only alphanumeric word tokens, entirely discarding punctuation and emojis. Consequently, a post that conveys anger solely through emojis provides no lexical features to the linear classifier. By contrast, a byte-level Byte-Pair Encoding (BPE) tokenizer preserves emojis as distinct tokens, making them available to the transformer. I tested this rather than assuming it: enabling BERTweet's emoji-to-text normalizer changed the out-of-fold macro F1 by −0.0018, which suggests the transformer was already using the raw emoji tokens.

### 1.3 Explanation of Your Model

**Ensemble architecture.** The model is a temperature-calibrated blend of transformer and linear components, all scored on the same five frozen folds. It was built in two stages: a first blend of three components (Section 3.5), and a second one that adds the emotion-trained checkpoints described in 1.2 (Sections 3.6 and 3.7). The numbers below are from the notebook run.

| Component | Out-of-Fold Macro F1 |
| --- | ---: |
| `twitter-roberta-base-2022-154m` (averaged over 3 seeds) | 0.6120 |
| `twitter-roberta-large-2022-154m` | 0.6101 |
| TF-IDF (word + char) + MiniLM with Linear SVM | 0.5708 |
| `j-hartmann/emotion-english-distilroberta-base` (averaged over 3 seeds) | 0.6262 |
| `j-hartmann/emotion-english-roberta-large` (averaged over 3 seeds) | 0.6242 |
| `SamLowe/roberta-base-go_emotions` | 0.6222 |
| `cardiffnlp/twitter-roberta-large-emotion-latest` | 0.6059 |
| `michellejieli/emotion_text_classifier` | 0.6014 |
| Three-component blend, Section 3.5 (nested CV estimate) | 0.6236 |
| Four-component blend: tweet base, two j-hartmann models (seed 42), linear (nested) | 0.6438 |
| **Seven-component blend (nested CV estimate)** | **0.6536** |

The first blend gains 0.0116 over its best component, about 17 test predictions. Three of the five emotion checkpoints beat the tweet-domain model on their own, by 0.010 to 0.014. The two that did not may be worth a guess, though I did not test either explanation: the cardiffnlp model was trained for multi-label detection with sigmoid outputs, so its head does not treat the classes as mutually exclusive, and michellejieli was trained on TV-show dialogue, which is further from tweets than the other sources.

Adding the two j-hartmann models to the three-component blend raised the nested estimate from 0.6236 to 0.6438, a gain of 0.0201 or about 29 test rows. That is the largest improvement after the move from linear models to transformers. The seven-component blend uses seed-averaged versions of the j-hartmann models plus the three other emotion checkpoints and reaches 0.6536, another 0.0098 or about 14 rows, higher in four of the five folds. Its weights are 0.20 each on the two j-hartmann models, go_emotions and the cardiffnlp large model, 0.10 each on the tweet-domain base and the linear model, and 0 on michellejieli, so the search dropped one component altogether.

**Multi-seed averaging.** Fine-tuning a transformer introduces stochastic variance due to the random initialization of the classification head and batch shuffling. Across three random seeds, the base model achieved out-of-fold macro F1 scores of 0.6015, 0.5995, and 0.6102, representing a spread of 0.0107. Averaging the logits across the three seeds stabilizes this variance and improves the score to 0.6120.

**Temperature calibration.** Because the linear SVM outputs uncalibrated decision margins and the transformers output logits, their raw predictions operate on different numerical scales. My first blending attempt skipped calibration, and it gave a weight of 0.85 to the weakest component (the linear SVM), simply because its uncalibrated softmax distribution was nearly uniform and needed a large multiplier to influence the combined output. Fitting a single temperature parameter per model by minimizing out-of-fold log loss (yielding temperatures of 1.50, 1.87, and 0.28, respectively) aligns the probability scales, allowing the subsequent weight optimization to reflect each model's true predictive contribution. The emotion checkpoints need the same treatment; their fitted temperatures range from 1.70 to 2.49.

**Nested cross-validation for ensemble weights.** Optimizing blending weights directly on out-of-fold predictions and evaluating on those same predictions introduces an optimistic bias, yielding an unnested score of 0.6288. To get an unbiased estimate, I used a nested cross-validation procedure: the ensemble weights were optimized on four folds and evaluated on the held-out fifth fold, rotating across all splits. This nested evaluation yields 0.6236, demonstrating that the unnested estimate overstated performance by 0.0052. For the four- and seven-component blends the gaps are 0.0032 (0.6470 against 0.6438) and 0.0019 (0.6555 against 0.6536). Nesting does not remove a second bias, though. I compared several combinations before settling on these seven components, and picking the best of several also flatters the winner, so I read the 0.0098 gain of the seven-component blend over the four-component one as a trend rather than a proven improvement.

**Impact of model capacity.** Scaling up from the 125M-parameter `roberta-base` model (0.6120) to the 355M-parameter `roberta-large` model (0.6101) decreased performance by 0.0019 (roughly 3 test instances), despite requiring 2.8 times the parameters and quadruple the training time. This diminishing return from increased model capacity was observed consistently across earlier experiments: in the Master notebook, a 544,964-parameter neural network underperformed a 16,764-parameter linear SVM on identical features, and a multi-layer perceptron trained on Word2Vec embeddings similarly failed to surpass a linear classifier. The emotion checkpoints repeat the pattern: the RoBERTa-large version (0.6242) is no better than the distilled one (0.6262), which has about 82M parameters. With only 3,360 labeled training instances, performance is constrained by sample size rather than model capacity, and increasing parameter counts yields no benefit. What did help was changing what the model knew before training, not how big it was.

**Error analysis.** Examining the confusion matrix of the linear model reveals systematic error patterns driven by class imbalance. Specifically, 24.2% of true disgust instances are misclassified as anger, whereas only 9.2% of anger instances are misclassified as disgust. Because anger has 48% more training samples than disgust, the decision boundary naturally favors the larger class in overlapping regions of the feature space. Similarly, the model exhibits a strong prediction bias toward the majority class joy, misclassifying 316 instances from other emotions into this category (952 predicted versus 875 actual). Under the macro F1 metric, these false negatives on minority classes heavily penalize the overall score.

A second consistent pattern emerges across the cross-validation splits: all four individual model runs recorded their lowest performance on Fold 4 (scoring 0.5779, 0.5715, 0.5915, and 0.5796, compared to 0.60–0.63 on the remaining folds). The fact that diverse architectures independently struggle on the exact same partition indicates that fold-to-fold variance is primarily driven by inherent data difficulty within specific splits rather than training instability. The emotion runs in Section 3.6 agree: eight of the nine also scored lowest on Fold 4, the exception being go_emotions, whose lowest fold was Fold 2.

**Validation reliability and leaderboard comparison.** Comparing the local out-of-fold estimates against five public leaderboard submissions illustrates the behavior of the validation strategy:

| Submission | Out-of-Fold | Public LB | Offset |
| --- | ---: | ---: | ---: |
| TF-IDF + char + MiniLM, Linear SVM | 0.5708 | 0.5672 | −0.0036 |
| Fine-tuned RoBERTa-base (single seed) | 0.6053 | 0.6233 | +0.0180 |
| Calibrated ensemble (initial version) | 0.6204 | 0.6364 | +0.0160 |
| Calibrated ensemble (with seed-averaged base) | 0.6216 | 0.6330 | +0.0114 |
| Calibrated ensemble, three components (Section 3.5) | 0.6236 | 0.6223 | −0.0013 |
| Four components, with two emotion checkpoints | 0.6424 | 0.6430 | +0.0006 |
| Seven components | 0.6495 | 0.6366 | −0.0129 |

The last two files were produced by standalone scripts that use the same method and settings as Sections 3.6 and 3.7, before I moved that code into this notebook. GPU fine-tuning is not reproducible bit for bit, so the notebook rerun gives 0.6438 and 0.6536 instead of 0.6424 and 0.6495, and its prediction files differ from the submitted ones in 57 and 68 of 1,440 rows.

Analyzing the top three ensemble submissions highlights the sampling variance of small test splits. Locally, the three ensembles differ by only 0.0032 across 3,360 rows (roughly 11 instances). On the public leaderboard, the same three submissions span a range of 0.0141, and their rank order is completely reversed. However, because the public leaderboard represents only 30% of the test set (approximately 432 rows), a 0.0141 difference corresponds to just 6 predictions. These minor rank reversals reflect random sampling noise on a small subset rather than meaningful performance differences. The two emotion blends did the same thing: the seven-component blend is 0.0071 higher out of fold, about 24 of 3,360 rows, and 0.0064 lower on the public leaderboard, about 3 of 432 rows. More importantly, local cross-validation accurately predicted the three major performance improvements: transitioning from linear models to fine-tuned transformers, combining diverse models into a calibrated ensemble, and starting from emotion-trained checkpoints (+0.0208 out of fold and +0.0100 on the public leaderboard, against the three-component blend with the seed-averaged base). I therefore ranked models by the 3,360-row out-of-fold estimate and treated the public leaderboard only as a formatting and sanity check. On that basis the seven-component blend comes first, and the four-component blend, which has the best public score, is the natural second of the two final entries Kaggle allows.

**Limitations.** Three known limitations remain in this study. First, the 16 duplicate text groups with contradictory labels impose an inherent ceiling on classification accuracy. Second, the 59 overlapping texts between the training and test sets allow models to memorize a small fraction of the test data, slightly inflating absolute scores across all competition entries. Finally, although the ensemble weights were evaluated via nested cross-validation, all hyperparameter and architectural choices were informed by the same 3,360 training instances; consequently, the reported 0.6536 macro F1 score should be interpreted as an estimate subject to roughly ±0.01 of fold-to-fold variance, rather than an exact point value.

### 1.4 AI Prompts, what and why you decide to use that technique that suggest by AI

I used Claude Code as a pair programmer for this competition. Instead of listing prompts word for word, I record the suggestions it made, the evidence each one produced on my own folds, and the decision I took. Several suggestions were rejected, and those are included here as well, because the evidence that something does not work was often more useful than the suggestions that did.

**Representation.** The first suggestion was to start from a word-level TF-IDF over unigrams and bigrams. I adopted it: on its own it reached an out-of-fold macro F1 of 0.4730, which gave me a sparse baseline that captures emotion vocabulary directly, and `sublinear_tf` kept posts of up to 508 words from dominating the vectors through sheer repetition. I then added character n-grams and MiniLM sentence embeddings, and kept both for the same reason: each one was measured on the same folds and each one paid for itself. Character n-grams reached 0.5045 alone, above the word view, and the two together reached 0.5146; adding MiniLM took that to 0.5708, a gain of 0.0562 or roughly 81 test rows, which is far larger than anything the classifier settings changed. The last suggestion in this group was to stop hand-building features and fine-tune a transformer pre-trained on 154 million tweets. I adopted that too, because it moved the out-of-fold score from 0.5708 to 0.6120, about 59 test rows, which is the single largest improvement in the whole project.

**Class imbalance.** Claude suggested turning on `class_weight="balanced"` for the linear model. I rejected it. Enabling it changed the out-of-fold score from 0.5096 to 0.5129, a difference of 0.0033, which is less than a fifth of the between-fold standard deviation of 0.0167 and therefore inside the range that random variation alone would produce. Later Claude suggested changing the decision rule itself, because `argmax` maximises accuracy and not macro F1. I tested two versions in separate script runs and rejected both. Six free per-class multipliers raised the nested score by only 0.0022, about 3 rows, winning on two folds and losing on two, and the fitted multipliers moved a lot between folds (the one for joy went from 0.52 to 1.05), which is what fitting noise looks like. A one-parameter version that divides the probabilities by the class prior raised to a power chose a power of 0.02 on every fold, so when it could not overfit, it decided to change almost nothing. I took that as a sign the temperature calibration had already done this job. Resampling was never tested, so I have no evidence about it either way.

**Model capacity and stability.** Claude suggested scaling the transformer up from 125M to 355M parameters. I rejected that as an upgrade: the larger model scored 0.6101 against the base model's 0.6120, which is 0.0019 lower, about 3 test rows, smaller than the fold-to-fold variation, and it cost 2.8 times the parameters and four times the training time. I did keep its predictions for the final ensemble, since a model that is not better on its own can still be wrong about different rows. I adopted the suggestion to train the base model under three random seeds and average them, because the three runs scored 0.6015, 0.5995 and 0.6102, a spread of 0.0107 that shows how much noise a single run carries; averaging them gave 0.6120. Two other suggestions were rejected after testing. Adding BERTweet to the ensemble moved the blend by 0.0002, which is not a result. Enabling BERTweet's emoji-to-text normaliser moved the out-of-fold score by -0.0018, and the reason that experiment failed is itself worth recording: a byte-level tokenizer already has tokens for emoji, so the gap only exists for the TF-IDF views, where the default token pattern discards them.

**Emotion-trained checkpoints.** The suggestion behind the largest gain in the second half of the project was to start from checkpoints already trained to recognise emotions and to keep their classification heads. I adopted it. The two j-hartmann models each beat the tweet-domain model on their own, and adding them to the blend raised the nested estimate by about 0.02 and gave my best public score, 0.6430. Not every follow-up worked, and I tested each one in separate script runs before deciding. A cardiffnlp base model trained for multi-label emotion detection scored only 0.6004 and was dropped. Two more seeds for each j-hartmann model improved each one by 0.005 to 0.008 on its own but moved the blend by 0.0007, about one row, because a blend already averages away much of the noise that seed averaging targets; the same happened with the tweet-domain base model. Four hyperparameter changes on the distilled model (maximum length 128, eight epochs, learning rate 5e-5, batch size 16) scored 0.6175, 0.6100, 0.6084 and 0.6172, all inside the 0.6162 to 0.6240 range the original settings produce when only the seed changes, so I kept the originals; longer training and the higher learning rate were both a little worse, which fits a small dataset that overfits easily. The three further checkpoints in the seven-component blend each added little or nothing when tried alone, but together they raised the nested estimate by 0.0049 in the script run, with three folds up and two down and a paired t of 1.18, and by 0.0098 in the notebook run. I submitted it and treat the gain as a trend.

**Using the test set without its labels.** Since every experiment pointed to the 3,360 training rows as the limit, Claude suggested pseudo-labelling: retrain with the test rows the ensemble is most confident about, using its own predictions as labels. The premise held, because training rows with confidence of at least 0.8 were 94.0% correct, against about 62% overall. The result still got worse, and it got worse the more rows I added: 0.6041, 0.6035 and 0.6021 at thresholds of 0.9, 0.8 and 0.7, against 0.6053 for the same model without pseudo-labels. Each difference alone is inside the noise, but all three go the same way. The class mix explains why. Among the confident rows at 0.8, joy made up 49.1% against 26.04% in the training data, and disgust only 2.4% against 13.51%. The model is most confident exactly where it is already biased, so feeding those rows back tells it the bias is right, and macro F1 punishes precisely that. I rejected it.

**The data itself.** The most important suggestion in this group was to split the folds with `StratifiedGroupKFold`, grouping by the MD5 hash of the normalised text. I adopted it. The training set contains 50 repeated texts covering 100 rows, and grouping by hash brought the number of groups spanning more than one fold to zero, which stops the model from being scored on rows it has already memorised, while class proportions stayed within 0.3 percentage points across the five folds. I also followed the suggestion to keep the 16 contradictory label groups rather than deleting them, because removing them would raise the validation score without making the model any better on new text. The one suggestion I rejected here was to override the 59 test texts that also appear in the training set with their known labels: the model already predicts 55 of the 59 correctly, so the override would have changed 4 rows out of 1,440 while adding a special case to the pipeline.

**How I judged whether something helped.** I adopted the suggestion to freeze a single five-fold split and reuse it everywhere. With each fold at roughly 672 rows and class proportions within 0.3 percentage points, every out-of-fold number in this report rests on exactly the same basis, from the linear models to the transformers, so a difference of a few predictions can be compared directly. Three related suggestions were also adopted. Fitting one temperature per model before searching for blend weights was necessary rather than cosmetic: without it the search put a weight of 0.85 on the weakest component purely to compensate for its flatter probability scale. Reporting the nested estimate rather than the direct one cost 0.0052 of apparent performance, 0.6236 against 0.6288, which is exactly the amount the direct number was flattering itself by. And the rule that a difference smaller than the between-fold standard deviation does not count is what rejected both the class weighting and the larger model, so it did real work rather than sitting in the notebook as a principle.

**Submissions.** I adopted the suggestion to spend one submission on a probe: a file predicting `joy` for all 1,440 rows, whose expected score could be worked out in advance under each candidate metric. It returned 0.0686, matching the macro F1 prediction of 0.0689 and ruling out micro at 0.2604 and weighted at 0.1076, so a single submission settled what the competition page did not state. I also produced both header variants, `id` and `post_id`, because the competition's own files disagreed about which was expected; `id` was accepted. The most consequential suggestion in this group was to select models on the out-of-fold estimate and treat the public leaderboard only as a sanity check, and the submission history shows why that was right: the three ensembles rank 0.6204, 0.6216, 0.6236 locally and 0.6364, 0.6330, 0.6223 on the public leaderboard, an exactly reversed order over a 432-row sample where the whole spread is 6 predictions. The emotion blends repeated it: the seven-component blend is 0.0071 higher out of fold than the four-component one and 0.0064 lower in public. Finally, every submission file is written by a function that runs five checks first, on column count, row count, label values, missing values and exact label spelling, so a malformed file cannot reach Kaggle.
---


**`From here on starts the code section for the competition.`**

---

# **Competition Code**

## 1. Preprocessing Steps

In [1]:
### Add the code related to the preprocessing steps in cells inside this section

# T0 step 1 - load the three competition files WITHOUT assuming any column name.
# The Kaggle Data page calls the label column "emotion" but the real header is "label",
# so every column list is printed and the printed names are what the rest of the code uses.
import json, re, time, hashlib, warnings
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_STATE = 42          # every randomised step reuses this one seed
N_SPLITS = 5               # one fixed 5-fold split is shared by every experiment
DATA_DIR = Path("data/kaggle")                 # competition files (kept out of git)
OUT_DIR = Path("../_workbench/kaggle")         # experiment records, never committed
for sub in ["experiments", "oof", "submissions"]:
    (OUT_DIR / sub).mkdir(parents=True, exist_ok=True)

with open(DATA_DIR / "final_posts.json", encoding="utf-8") as fh:
    raw_posts = json.load(fh)
print("final_posts.json is a", type(raw_posts).__name__, "of length", len(raw_posts))
print("keys of the first record:", list(raw_posts[0].keys()))

posts = pd.DataFrame(raw_posts)
identification = pd.read_csv(DATA_DIR / "data_identification.csv")
emotion = pd.read_csv(DATA_DIR / "emotion.csv")
sample_submission = pd.read_csv(DATA_DIR / "sampleSubmission.csv")

for name, frame in [("posts", posts), ("identification", identification),
                    ("emotion", emotion), ("sample_submission", sample_submission)]:
    print(f"{name:18s} shape={str(frame.shape):12s} columns={list(frame.columns)}")

print("\nposts.post_id is unique:", posts["post_id"].is_unique)
print("missing values in posts:", posts.isna().sum().to_dict())
print("train/test split recorded in data_identification.csv:",
      identification["set"].value_counts().to_dict())


final_posts.json is a list of length 4800
keys of the first record: ['post_id', 'text']
posts              shape=(4800, 2)    columns=['post_id', 'text']
identification     shape=(4800, 2)    columns=['post_id', 'set']
emotion            shape=(3360, 2)    columns=['post_id', 'label']
sample_submission  shape=(1440, 2)    columns=['post_id', 'label']

posts.post_id is unique: True
missing values in posts: {'post_id': 0, 'text': 0}
train/test split recorded in data_identification.csv: {'train': 3360, 'test': 1440}


In [2]:
# T0 step 2 - merge the three files on post_id and assert the structure we were promised.
# A silent merge failure here would poison everything downstream, so each expectation
# is checked with an assert rather than just printed and eyeballed.
merged = posts.merge(identification, on="post_id", how="outer", indicator=True)
print("merge coverage posts vs identification:", merged["_merge"].value_counts().to_dict())
assert (merged["_merge"] == "both").all(), "some post_id appears in only one of the two files"

merged = merged.drop(columns="_merge").merge(emotion, on="post_id", how="left")
train = merged[merged["set"] == "train"].reset_index(drop=True)
test = merged[merged["set"] == "test"].reset_index(drop=True)

print(f"\ntrain rows {len(train)} + test rows {len(test)} = {len(train) + len(test)}")
assert len(train) + len(test) == 4800, "train + test should cover all 4,800 posts"
assert train["label"].notna().all(), "every training row must carry a label"
assert test["label"].isna().all(), "no test row may carry a label"
assert len(test) == len(sample_submission), "test rows must match sampleSubmission rows"
assert set(test["post_id"]) == set(sample_submission["post_id"]), "test ids must match the sample file"
print("test ids match sampleSubmission as a set:", True)
print("test ids are in the same ORDER as sampleSubmission:",
      list(test["post_id"]) == list(sample_submission["post_id"]))
print("-> the submission must therefore be re-ordered to the sample file, not written in test order")

LABELS = sorted(train["label"].unique())
print("\nthe six labels, exactly as spelled in emotion.csv:", LABELS)


merge coverage posts vs identification: {'both': 4800, 'left_only': 0, 'right_only': 0}

train rows 3360 + test rows 1440 = 4800
test ids match sampleSubmission as a set: True
test ids are in the same ORDER as sampleSubmission: False
-> the submission must therefore be re-ordered to the sample file, not written in test order

the six labels, exactly as spelled in emotion.csv: ['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise']


In [3]:
# T0 step 3 - describe the data, and turn the class distribution into the three numbers
# that the probe submission will be read against.
# Mean F1 is not defined anywhere on the competition pages (macro / micro / weighted all
# fit the words), so one throw-away submission that predicts a single class everywhere
# will tell us which one it is. If the largest class covers a share p of the scored subset,
# that submission gets precision = p, recall = 1, so its F1 for that class is 2p/(1+p),
# and the three candidate metrics give three very different totals.
counts = train["label"].value_counts()
distribution = pd.DataFrame({
    "n": counts,
    "share_%": (counts / len(train) * 100).round(2),
})
print("=== class distribution in the training set ===")
print(distribution.to_string())
print("\nsmallest class:", counts.idxmin(), counts.min(),
      "| largest class:", counts.idxmax(), counts.max(),
      f"| imbalance ratio {counts.max() / counts.min():.2f}x")

MAJORITY_LABEL = counts.idxmax()
p = counts.max() / len(train)
f1_of_that_class = 2 * p / (1 + p)
probe_expectation = {
    "macro": f1_of_that_class / 6,
    "micro": p,
    "weighted": p * f1_of_that_class,
}
print(f"\n=== what a constant '{MAJORITY_LABEL}' submission should score (p = {p:.4f}) ===")
for metric, value in probe_expectation.items():
    print(f"  if Mean F1 means {metric:9s} -> {value:.4f}")
print(f"  weighted / macro = {probe_expectation['weighted'] / probe_expectation['macro']:.2f}x,"
      " far enough apart to decide from a single submission")
print("  caveat: p is estimated from the training split; the public subset is only 30% of the"
      " test set and its true class shares are unknown.")

train_lengths = pd.DataFrame({
    "chars": train["text"].str.len(),
    "words": train["text"].str.split().str.len(),
})
print("\n=== text length, training set ===")
print(train_lengths.describe().round(2).to_string())
print("\n=== words per post, by class ===")
print(train.assign(words=train_lengths["words"])
          .groupby("label")["words"].agg(["mean", "median", "max"]).round(2).to_string())
print("\n-> the length distribution is heavily right-skewed (median 13 words, maximum above 500),"
      " which is why TF-IDF is used with sublinear term frequencies below.")


=== class distribution in the training set ===
            n  share_%
label                 
joy       875    26.04
anger     674    20.06
sadness   628    18.69
disgust   454    13.51
surprise  441    13.12
fear      288     8.57

smallest class: fear 288 | largest class: joy 875 | imbalance ratio 3.04x

=== what a constant 'joy' submission should score (p = 0.2604) ===
  if Mean F1 means macro     -> 0.0689
  if Mean F1 means micro     -> 0.2604
  if Mean F1 means weighted  -> 0.1076
  weighted / macro = 1.56x, far enough apart to decide from a single submission
  caveat: p is estimated from the training split; the public subset is only 30% of the test set and its true class shares are unknown.

=== text length, training set ===
         chars    words
count  3360.00  3360.00
mean     87.78    16.51
std     121.88    24.96
min       5.00     1.00
25%      42.00     8.00
50%      68.00    13.00
75%     101.25    19.00
max    2545.00   508.00

=== words per post, by class ===
         

In [4]:
# T0 step 4 - build the ONE fixed cross-validation split that every later experiment reuses.
# The four steps below must happen in this order:
#   1. normalise the text (lowercase, collapse whitespace)
#   2. count duplicate texts
#   3. if duplicates exist, group them so that copies of one text cannot be split across folds
#   4. only then write folds.csv
# Skipping step 3 would let the same sentence sit in both the training and the validation
# fold, which inflates every out-of-fold score and makes all later comparisons worthless.
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold

def normalise(text):
    """Lowercase and collapse runs of whitespace - used only for duplicate detection."""
    return re.sub(r"\s+", " ", str(text).lower()).strip()

train["text_normalised"] = train["text"].map(normalise)
test["text_normalised"] = test["text"].map(normalise)

train_dupes = train["text_normalised"].value_counts()
train_dupes = train_dupes[train_dupes > 1]
test_dupes = test["text_normalised"].value_counts()
test_dupes = test_dupes[test_dupes > 1]
label_disagreement = train.groupby("text_normalised")["label"].nunique()

print("=== duplicate texts after normalisation ===")
print(f"train: {len(train_dupes)} distinct texts appear more than once, covering"
      f" {int(train_dupes.sum())} rows (largest group {int(train_dupes.max())} rows)")
print(f"test : {len(test_dupes)} distinct texts appear more than once, covering"
      f" {int(test_dupes.sum())} rows")
print(f"identical training texts carrying DIFFERENT labels: {(label_disagreement > 1).sum()}")
print("-> those contradictory pairs put a ceiling on any achievable score, and they are the"
      " reason the fold split has to be grouped.")

overlap = set(train["text_normalised"]) & set(test["text_normalised"])
print(f"\ntexts that occur in BOTH train and test: {len(overlap)} distinct"
      f" ({train['text_normalised'].isin(overlap).sum()} train rows,"
      f" {test['text_normalised'].isin(overlap).sum()} test rows)")
print("-> recorded as a limitation in the report: part of the test set is memorisable.")

train["group"] = train["text_normalised"].map(
    lambda s: hashlib.md5(s.encode("utf-8")).hexdigest()
)
if len(train_dupes) > 0:
    splitter = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    split_iter = splitter.split(train, train["label"], groups=train["group"])
    print("\nduplicates found -> using StratifiedGroupKFold with the text hash as the group")
else:
    splitter = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    split_iter = splitter.split(train, train["label"])
    print("\nno duplicates -> plain StratifiedKFold is enough")

train["fold"] = -1
for fold_id, (_, validation_index) in enumerate(split_iter):
    train.loc[validation_index, "fold"] = fold_id
assert (train["fold"] >= 0).all(), "every training row must land in exactly one fold"

print("\nrows per fold:", train["fold"].value_counts().sort_index().to_dict())
print("groups that leaked across more than one fold:",
      int((train.groupby("group")["fold"].nunique() > 1).sum()), "(must be 0)")
print("\nclass share per fold, in % (stratification check):")
print((pd.crosstab(train["fold"], train["label"], normalize="index") * 100).round(2).to_string())

train[["post_id", "label", "fold"]].to_csv(OUT_DIR / "folds.csv", index=False)
print(f"\nwritten: {OUT_DIR / 'folds.csv'} - this file is now frozen for the whole competition")


=== duplicate texts after normalisation ===
train: 50 distinct texts appear more than once, covering 100 rows (largest group 2 rows)
test : 8 distinct texts appear more than once, covering 16 rows
identical training texts carrying DIFFERENT labels: 16
-> those contradictory pairs put a ceiling on any achievable score, and they are the reason the fold split has to be grouped.

texts that occur in BOTH train and test: 59 distinct (61 train rows, 59 test rows)
-> recorded as a limitation in the report: part of the test set is memorisable.

duplicates found -> using StratifiedGroupKFold with the text hash as the group

rows per fold: {0: 673, 1: 672, 2: 672, 3: 672, 4: 671}
groups that leaked across more than one fold: 0 (must be 0)

class share per fold, in % (stratification check):
label  anger  disgust  fear    joy  sadness  surprise
fold                                                 
0      20.06    13.52  8.62  26.00    18.72     13.08
1      20.09    13.54  8.48  26.04    18.75    

## 2. Feature Engineering Steps

In [5]:
### Add the code related to the feature engineering steps in cells inside this section

# Feature family 1 and 2 - two TF-IDF views of the same text, horizontally stacked.
#   word level  (1,2)-grams: catches vocabulary and short phrases such as "not happy"
#   char level  char_wb (2,5)-grams: catches lengthened words ("angryyy"), misspellings and
#               emoji, which the word view throws away as unseen tokens
# Both vectorisers are fitted INSIDE each fold on the training part only. Fitting them once
# on all 3,360 rows would let validation-fold vocabulary and document frequencies leak into
# the features and would quietly inflate every score.
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer

WORD_TFIDF = dict(ngram_range=(1, 2), min_df=2, sublinear_tf=True, strip_accents="unicode")
CHAR_TFIDF = dict(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True)

# A one-off look at how wide each view is when fitted on four of the five folds.
demo_train_text = train.loc[train["fold"] != 0, "text"].values
for view_name, settings in [("word", WORD_TFIDF), ("char_wb", CHAR_TFIDF)]:
    matrix = TfidfVectorizer(**settings).fit_transform(demo_train_text)
    density = matrix.nnz / (matrix.shape[0] * matrix.shape[1]) * 100
    print(f"{view_name:8s} view: {matrix.shape[1]:6d} features, {density:.3f}% of entries non-zero")
print("\n-> the character view is roughly four times wider than the word view and is the"
      " cheapest single improvement available at this stage.")


word     view:   6107 features, 0.263% of entries non-zero
char_wb  view:  24566 features, 0.755% of entries non-zero

-> the character view is roughly four times wider than the word view and is the cheapest single improvement available at this stage.


In [6]:
# Feature family 3 - MiniLM sentence embeddings.
# This is the same all-MiniLM-L6-v2 model used in Master section 5, so no new download is
# needed. Unlike TF-IDF it carries meaning learned from a large external corpus, so two posts
# can be close even with no shared words; that is exactly what the sparse views cannot do.
# The encoder is UNSUPERVISED with respect to our labels, so encoding all 4,800 texts once -
# outside the fold loop - leaks nothing, and it keeps the cross-validation cheap.
from sentence_transformers import SentenceTransformer

EMBEDDING_CACHE = OUT_DIR / "minilm_embeddings.npz"
if EMBEDDING_CACHE.exists():
    cached = np.load(EMBEDDING_CACHE)
    train_embeddings, test_embeddings = cached["train"], cached["test"]
    print("loaded cached embeddings from", EMBEDDING_CACHE)
else:
    encoder = SentenceTransformer("all-MiniLM-L6-v2")
    started = time.time()
    train_embeddings = encoder.encode(list(train["text"]), batch_size=64,
                                      normalize_embeddings=True, show_progress_bar=False)
    test_embeddings = encoder.encode(list(test["text"]), batch_size=64,
                                     normalize_embeddings=True, show_progress_bar=False)
    np.savez(EMBEDDING_CACHE, train=train_embeddings, test=test_embeddings)
    print(f"encoded {len(train) + len(test)} texts in {time.time() - started:.0f}s"
          f" and cached them to {EMBEDDING_CACHE}")

print("train embeddings", train_embeddings.shape, "| test embeddings", test_embeddings.shape)
print("rows are L2-normalised:", np.allclose(np.linalg.norm(train_embeddings, axis=1), 1.0))

EMBEDDING_SCALE = 1.0   # embeddings sit beside TF-IDF columns; this keeps the two blocks comparable

def build_features(train_text, apply_text, train_embedding, apply_embedding, views):
    """Fit the requested views on train_text only, then apply them to apply_text."""
    fitted_train, fitted_apply = [], []
    if "word" in views:
        vectoriser = TfidfVectorizer(**WORD_TFIDF)
        fitted_train.append(vectoriser.fit_transform(train_text))
        fitted_apply.append(vectoriser.transform(apply_text))
    if "char" in views:
        vectoriser = TfidfVectorizer(**CHAR_TFIDF)
        fitted_train.append(vectoriser.fit_transform(train_text))
        fitted_apply.append(vectoriser.transform(apply_text))
    if "emb" in views:
        fitted_train.append(csr_matrix(train_embedding * EMBEDDING_SCALE))
        fitted_apply.append(csr_matrix(apply_embedding * EMBEDDING_SCALE))
    if len(fitted_train) == 1:
        return fitted_train[0], fitted_apply[0]
    return hstack(fitted_train).tocsr(), hstack(fitted_apply).tocsr()

print("\nbuild_features() is the single place where features are made, so the"
      " cross-validation below and the final full-data fit cannot drift apart.")


C:\MyProjects\DM\Lab-2\kaggle_gpu\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


loaded cached embeddings from ..\_workbench\kaggle\minilm_embeddings.npz
train embeddings (3360, 384) | test embeddings (1440, 384)
rows are L2-normalised: True

build_features() is the single place where features are made, so the cross-validation below and the final full-data fit cannot drift apart.


## 3. Model Implementation Steps

In [7]:
### Add the code related to the model implementation steps in cells inside this section

# Model search - out-of-fold macro F1 over the frozen 5-fold split from section 1.
# Every configuration is scored on exactly the same folds, so the numbers are comparable and
# the out-of-fold predictions could later be reused for an ensemble.
# Three things are varied, one at a time: which feature views are used, whether the classes
# are reweighted, and the regularisation strength C.
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, classification_report, confusion_matrix

warnings.filterwarnings("ignore", category=UserWarning)

y_train = train["label"].values
fold_ids = train["fold"].values
texts = train["text"].values

def evaluate(views, class_weight, C):
    """Return out-of-fold predictions and the three candidate F1 averages."""
    out_of_fold = np.empty(len(train), dtype=object)
    per_fold = []
    started = time.time()
    for fold_id in range(N_SPLITS):
        is_train = fold_ids != fold_id
        is_validation = fold_ids == fold_id
        X_train, X_validation = build_features(
            texts[is_train], texts[is_validation],
            train_embeddings[is_train], train_embeddings[is_validation], views,
        )
        model = LinearSVC(C=C, class_weight=class_weight,
                          random_state=RANDOM_STATE, max_iter=10000)
        model.fit(X_train, y_train[is_train])
        predictions = model.predict(X_validation)
        out_of_fold[is_validation] = predictions
        per_fold.append(f1_score(y_train[is_validation], predictions, average="macro"))
    return out_of_fold, {
        "views": "+".join(views),
        "class_weight": str(class_weight),
        "C": C,
        "oof_macro_f1": f1_score(y_train, out_of_fold, average="macro"),
        "oof_micro_f1": f1_score(y_train, out_of_fold, average="micro"),
        "oof_weighted_f1": f1_score(y_train, out_of_fold, average="weighted"),
        "fold_mean": float(np.mean(per_fold)),
        "fold_std": float(np.std(per_fold)),
        "seconds": round(time.time() - started, 1),
    }

CANDIDATES = [
    (["word"],                "balanced", 0.10),   # single sparse view, for reference
    (["char"],                "balanced", 0.10),   # characters alone already beat words
    (["emb"],                 "balanced", 1.00),   # dense view alone
    (["word", "char"],        None,       0.10),   # does reweighting help?
    (["word", "char"],        "balanced", 0.10),
    (["word", "char"],        "balanced", 0.20),   # does less regularisation help?
    (["word", "char", "emb"], "balanced", 0.05),
    (["word", "char", "emb"], "balanced", 0.10),
]

records, oof_store = [], {}
for views, class_weight, C in CANDIDATES:
    predictions, record = evaluate(views, class_weight, C)
    key = f"{record['views']}|cw={record['class_weight']}|C={C}"
    oof_store[key] = predictions
    records.append(record)
    print(f"{key:40s} macro={record['oof_macro_f1']:.4f} (folds "
          f"{record['fold_mean']:.4f} +/- {record['fold_std']:.4f})  "
          f"micro={record['oof_micro_f1']:.4f}  weighted={record['oof_weighted_f1']:.4f}  "
          f"{record['seconds']}s", flush=True)

results = pd.DataFrame(records).sort_values("oof_macro_f1", ascending=False)
results.to_csv(OUT_DIR / "experiments" / "t1_grid.csv", index=False)
print("\n=== ranked by out-of-fold macro F1 ===")
print(results[["views", "class_weight", "C", "oof_macro_f1", "fold_std",
               "oof_micro_f1", "oof_weighted_f1"]].to_string(index=False))

best = results.iloc[0]
print(f"\nbest configuration: views={best['views']}, class_weight={best['class_weight']},"
      f" C={best['C']} -> out-of-fold macro F1 {best['oof_macro_f1']:.4f}"
      f" (fold standard deviation {best['fold_std']:.4f})")
print("the top configurations sit within one fold standard deviation of each other, so the"
      " ranking inside that group is noise; what is NOT noise is the gap opened by adding"
      " the embedding view.")


word|cw=balanced|C=0.1                   macro=0.4730 (folds 0.4728 +/- 0.0134)  micro=0.4940  weighted=0.4915  0.4s
char|cw=balanced|C=0.1                   macro=0.5045 (folds 0.5041 +/- 0.0116)  micro=0.5229  weighted=0.5201  1.7s
emb|cw=balanced|C=1.0                    macro=0.4863 (folds 0.4856 +/- 0.0091)  micro=0.5071  weighted=0.5075  1.9s
word+char|cw=None|C=0.1                  macro=0.5096 (folds 0.5086 +/- 0.0188)  micro=0.5357  weighted=0.5248  1.9s
word+char|cw=balanced|C=0.1              macro=0.5129 (folds 0.5122 +/- 0.0133)  micro=0.5280  weighted=0.5264  1.9s
word+char|cw=balanced|C=0.2              macro=0.5146 (folds 0.5138 +/- 0.0159)  micro=0.5307  weighted=0.5292  2.3s
word+char+emb|cw=balanced|C=0.05         macro=0.5708 (folds 0.5705 +/- 0.0167)  micro=0.5878  weighted=0.5851  3.6s
word+char+emb|cw=balanced|C=0.1          macro=0.5657 (folds 0.5654 +/- 0.0154)  micro=0.5839  weighted=0.5819  3.5s

=== ranked by out-of-fold macro F1 ===
        views class_weig

In [8]:
# Refit the chosen configuration on all 3,360 labelled rows and predict the test set.
# The vectorisers are fitted here on the full training text - that is correct, because the
# test labels are never touched; only the fold-level fits had to stay inside the fold.
BEST_VIEWS = ["word", "char", "emb"]
BEST_CLASS_WEIGHT = "balanced"
BEST_C = 0.05

X_full, X_test = build_features(train["text"].values, test["text"].values,
                                train_embeddings, test_embeddings, BEST_VIEWS)
print("full training matrix", X_full.shape, "| test matrix", X_test.shape)

final_model = LinearSVC(C=BEST_C, class_weight=BEST_CLASS_WEIGHT,
                        random_state=RANDOM_STATE, max_iter=10000)
final_model.fit(X_full, y_train)
test_predictions = final_model.predict(X_test)

print("\npredicted class distribution on the test set, next to the training distribution:")
comparison = pd.DataFrame({
    "test_predicted_%": (pd.Series(test_predictions).value_counts(normalize=True) * 100).round(2),
    "train_actual_%": (train["label"].value_counts(normalize=True) * 100).round(2),
})
print(comparison.to_string())
print("\n-> class_weight='balanced' pushes the rare classes up, so the predicted shares are"
      " flatter than the training shares. That is deliberate: macro F1 rewards the small"
      " classes as much as the large ones.")

# Keep the out-of-fold predictions of the chosen configuration for the report and for any
# later ensemble work.
best_key = f"word+char+emb|cw={BEST_CLASS_WEIGHT}|C={BEST_C}"
oof_frame = pd.DataFrame({
    "post_id": train["post_id"], "true_label": y_train,
    "oof_prediction": oof_store[best_key], "fold": fold_ids,
})
oof_frame.to_csv(OUT_DIR / "oof" / "tfidf_char_minilm_linearsvc.csv", index=False)

print("\n=== per-class out-of-fold performance of the chosen configuration ===")
print(classification_report(y_train, oof_store[best_key], digits=3))
print("confusion matrix (rows = true label, columns = predicted):")
print(pd.DataFrame(confusion_matrix(y_train, oof_store[best_key], labels=LABELS),
                   index=LABELS, columns=LABELS).to_string())


full training matrix (3360, 35420) | test matrix (1440, 35420)

predicted class distribution on the test set, next to the training distribution:
          test_predicted_%  train_actual_%
anger                22.01           20.06
disgust              10.83           13.51
fear                  8.61            8.57
joy                  28.68           26.04
sadness              16.88           18.69
surprise             12.99           13.12

-> class_weight='balanced' pushes the rare classes up, so the predicted shares are flatter than the training shares. That is deliberate: macro F1 rewards the small classes as much as the large ones.

=== per-class out-of-fold performance of the chosen configuration ===
              precision    recall  f1-score   support

       anger      0.545     0.580     0.562       674
     disgust      0.507     0.421     0.460       454
        fear      0.609     0.580     0.594       288
         joy      0.668     0.727     0.696       875
     sadness

In [9]:
# Write the submission files, and refuse to write anything that fails the five checks.
# Two header spellings are produced on purpose: the Overview page and the Kaggle API both
# say the identifier column is "id", while the downloaded sampleSubmission.csv header says
# "post_id". The probe submission is uploaded first and settles which one Kaggle accepts.
SUBMISSION_DIR = OUT_DIR / "submissions"

def check_and_write(post_ids, labels, stem):
    """Run the five pre-submission checks, then write the id and post_id header variants."""
    frame = pd.DataFrame({"post_id": post_ids, "label": labels})
    # the scored file must follow sampleSubmission's row order, so join onto it
    frame = sample_submission[["post_id"]].merge(frame, on="post_id", how="left")

    problems = []
    if list(frame.columns) != ["post_id", "label"]:
        problems.append(f"1. expected exactly two columns, got {list(frame.columns)}")
    if len(frame) != len(sample_submission):
        problems.append(f"2. expected {len(sample_submission)} rows, got {len(frame)}")
    unexpected = set(frame["label"].dropna()) - set(LABELS)
    if unexpected:
        problems.append(f"3. labels outside the six classes: {unexpected}")
    if frame.isna().any().any():
        problems.append(f"4. missing values: {frame.isna().sum().to_dict()}")
    if not set(frame["label"].dropna()) <= set(emotion["label"].unique()):
        problems.append("5. label strings do not match emotion.csv exactly")
    if problems:
        raise ValueError("submission rejected:\n  " + "\n  ".join(problems))

    written = []
    for id_column in ["id", "post_id"]:
        out = frame.rename(columns={"post_id": id_column})
        path = SUBMISSION_DIR / f"{stem}__{id_column}.csv"
        out.to_csv(path, index=False)
        written.append(path)
    print(f"all five checks passed for '{stem}' "
          f"({len(frame)} rows, {frame['label'].nunique()} distinct labels)")
    for path in written:
        print("   written:", path)
    return frame

# Submission A - the probe. Predicts the majority class everywhere, purely to read off which
# average "Mean F1" refers to. It must never be chosen as a final submission.
probe = check_and_write(test["post_id"].values,
                        np.full(len(test), MAJORITY_LABEL), "probe_constant")
print(f"\nPROBE: every row predicted '{MAJORITY_LABEL}'. Expected score if the metric is"
      f" macro {probe_expectation['macro']:.4f} / micro {probe_expectation['micro']:.4f}"
      f" / weighted {probe_expectation['weighted']:.4f}")

# Submission B - the real first entry.
submission = check_and_write(test["post_id"].values, test_predictions, "t1_tfidf_char_minilm")
print(f"\nREAL ENTRY: out-of-fold macro F1 {best['oof_macro_f1']:.4f};"
      " the first three rows of the file are")
print(submission.head(3).to_string(index=False))

print("\nUpload order: the probe first (it settles both the metric and the header), then the"
      " real entry. Both go in the submission ledger before anything is uploaded.")


all five checks passed for 'probe_constant' (1440 rows, 1 distinct labels)
   written: ..\_workbench\kaggle\submissions\probe_constant__id.csv
   written: ..\_workbench\kaggle\submissions\probe_constant__post_id.csv

PROBE: every row predicted 'joy'. Expected score if the metric is macro 0.0689 / micro 0.2604 / weighted 0.1076
all five checks passed for 't1_tfidf_char_minilm' (1440 rows, 6 distinct labels)
   written: ..\_workbench\kaggle\submissions\t1_tfidf_char_minilm__id.csv
   written: ..\_workbench\kaggle\submissions\t1_tfidf_char_minilm__post_id.csv

REAL ENTRY: out-of-fold macro F1 0.5708; the first three rows of the file are
post_id   label
xyzx9jp   anger
l3vzzxt sadness
g680iug sadness

Upload order: the probe first (it settles both the metric and the header), then the real entry. Both go in the submission ledger before anything is uploaded.


### 3.4 Fine-tuning a tweet-domain transformer

The models above all take a fixed representation and fit a classifier on top of it. This section changes the representation itself: a transformer pre-trained on 154 million tweets, fine-tuned end to end on our 3,360 labelled ones, on the same frozen folds so the score is comparable with everything above.

**This section needs the GPU kernel** (`Python (lab2-kaggle-gpu)`). Five folds take about two minutes on an RTX 5090; the whole section, including three seeds and the larger model, takes roughly fifteen.

In [10]:
# Fine-tuning helpers. This section needs the GPU kernel (Python (lab2-kaggle-gpu)):
# the linear models above are seconds of CPU work, but fine-tuning a transformer five
# times is not. Everything else about the experiment is unchanged - the same frozen
# folds, the same macro F1, the same comparison - so these numbers sit beside the
# linear ones in the table above rather than replacing them.
import time

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT AVAILABLE")

LABEL_TO_INDEX = {label: i for i, label in enumerate(LABELS)}
y_index = np.array([LABEL_TO_INDEX[label] for label in y_train])


def tweet_normalise(text):
    """What the cardiffnlp tweet models were pre-trained on: handles and links masked.
    Emoji are left alone on purpose - a byte-level tokenizer has tokens for them, so
    unlike the TF-IDF views above this representation can actually see them."""
    text = re.sub(r"@\w+", "@user", str(text))
    text = re.sub(r"http\S+", "http", text)
    return " ".join(text.split())


class TweetDataset(Dataset):
    """Tokenise once, at a fixed length, so every batch has the same shape."""

    def __init__(self, texts, labels, tokeniser, max_len):
        self.encoded = tokeniser([tweet_normalise(t) for t in texts], truncation=True,
                                 max_length=max_len, padding="max_length", return_tensors="pt")
        self.labels = None if labels is None else torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return self.encoded["input_ids"].shape[0]

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.encoded.items()}
        if self.labels is not None:
            item["labels"] = self.labels[i]
        return item


@torch.no_grad()
def predict_logits(model, loader):
    model.eval()
    out = []
    for batch in loader:
        batch = {k: v.cuda() for k, v in batch.items() if k != "labels"}
        with torch.autocast("cuda", dtype=torch.bfloat16):
            out.append(model(**batch).logits.float().cpu())
    return torch.cat(out).numpy()


def finetune_cv(model_name, seed=RANDOM_STATE, epochs=5, lr=3e-5, batch_size=32, max_len=96):
    """Fine-tune model_name once per fold of the frozen split.

    Returns out-of-fold logits for the 3,360 training rows and the average of the five
    folds' logits for the 1,440 test rows. Averaging the folds is a small ensemble for
    free: five models trained on overlapping four-fifths of the data, which is steadier
    than any one of them.
    """
    torch.manual_seed(seed)
    np.random.seed(seed)
    oof = np.zeros((len(train), len(LABELS)), dtype=np.float32)
    test_logits = np.zeros((len(test), len(LABELS)), dtype=np.float32)
    started = time.time()

    for fold in range(N_SPLITS):
        is_validation = fold_ids == fold
        tokeniser = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForSequenceClassification.from_pretrained(
            model_name, num_labels=len(LABELS), ignore_mismatched_sizes=True).cuda()

        loader = DataLoader(TweetDataset(texts[~is_validation], y_index[~is_validation],
                                         tokeniser, max_len),
                            batch_size=batch_size, shuffle=True)
        optimiser = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
        schedule = torch.optim.lr_scheduler.OneCycleLR(
            optimiser, max_lr=lr, total_steps=len(loader) * epochs, pct_start=0.1)

        model.train()
        for _ in range(epochs):
            for batch in loader:
                batch = {k: v.cuda() for k, v in batch.items()}
                with torch.autocast("cuda", dtype=torch.bfloat16):
                    logits = model(**{k: v for k, v in batch.items() if k != "labels"}).logits
                    loss = F.cross_entropy(logits, batch["labels"])
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimiser.step()
                schedule.step()
                optimiser.zero_grad(set_to_none=True)

        oof[is_validation] = predict_logits(
            model, DataLoader(TweetDataset(texts[is_validation], None, tokeniser, max_len),
                              batch_size=128))
        test_logits += predict_logits(
            model, DataLoader(TweetDataset(test["text"].values, None, tokeniser, max_len),
                              batch_size=128)) / N_SPLITS
        print(f"  fold {fold}: macro F1 "
              f"{f1_score(y_index[is_validation], oof[is_validation].argmax(1), average='macro'):.4f}",
              flush=True)
        del model
        torch.cuda.empty_cache()

    score = f1_score(y_index, oof.argmax(1), average="macro")
    print(f"  out-of-fold macro F1 {score:.4f}  ({time.time() - started:.0f}s)", flush=True)
    return oof, test_logits, score


GPU: NVIDIA GeForce RTX 5090 Laptop GPU


In [11]:
# The tweet-domain base model, trained three times with different seeds.
#
# Why a tweet model rather than a general one: the same argument the Master notebook made
# for MiniLM in Section 5. Our corpus is 3,360 tweets, which is far too little to learn
# what words mean; this checkpoint was pre-trained on 154 million of them.
#
# Why three seeds: a single fine-tuning run carries its own randomness - the classifier
# head starts from scratch and the batches are shuffled. Averaging three runs cancels
# part of that, and the spread between them measures how much it was worth cancelling.
BASE_MODEL = "cardiffnlp/twitter-roberta-base-2022-154m"
SEEDS = [RANDOM_STATE, 7, 2026]

base_oof, base_test, base_scores = [], [], []
for seed in SEEDS:
    print(f"{BASE_MODEL}  seed={seed}")
    oof, test_logits, score = finetune_cv(BASE_MODEL, seed=seed)
    base_oof.append(oof)
    base_test.append(test_logits)
    base_scores.append(score)

base_avg_oof = np.mean(base_oof, axis=0)
base_avg_test = np.mean(base_test, axis=0)
base_avg_score = f1_score(y_index, base_avg_oof.argmax(1), average="macro")

print(f"\nthe three seeds on their own : {[round(s, 4) for s in base_scores]}"
      f"  (spread {max(base_scores) - min(base_scores):.4f})")
print(f"averaged over the three seeds: {base_avg_score:.4f}"
      f"  ({base_avg_score - max(base_scores):+.4f} on the best single seed)")
print(f"the best linear model above  : {best['oof_macro_f1']:.4f}"
      f"  -> fine-tuning gains {base_avg_score - best['oof_macro_f1']:+.4f},"
      f" about {(base_avg_score - best['oof_macro_f1']) * len(test):.0f} of the {len(test)} test rows")


cardiffnlp/twitter-roberta-base-2022-154m  seed=42


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 32830.49it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-2022-154m
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  fold 0: macro F1 0.5988


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 28151.61it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-2022-154m
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  fold 1: macro F1 0.6066


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 24625.32it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-2022-154m
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


... (21 outputs left out)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 28135.31it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-2022-154m
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  fold 3: macro F1 0.6249


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 24620.91it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-2022-154m
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  fold 4: macro F1 0.5915
  out-of-fold macro F1 0.6102  (188s)

the three seeds on their own : [0.6015, 0.5995, 0.6102]  (spread 0.0107)
averaged over the three seeds: 0.6120  (+0.0018 on the best single seed)
the best linear model above  : 0.5708  -> fine-tuning gains +0.0412, about 59 of the 1440 test rows


In [12]:
# The same architecture at 355 million parameters instead of 125 million, to find out
# whether capacity is what this problem is short of. Section 7 of the Master notebook
# asked the same question with a 544,964-weight network against a 16,764-weight linear
# SVM and the answer was no; this is that question asked again at a scale where the
# model is not learning the language from scratch.
LARGE_MODEL = "cardiffnlp/twitter-roberta-large-2022-154m"

print(LARGE_MODEL)
large_oof, large_test, large_score = finetune_cv(LARGE_MODEL, epochs=5, lr=1.5e-5, batch_size=16)

print(f"\nlarge  {large_score:.4f}   base (seed-averaged) {base_avg_score:.4f}"
      f"   difference {large_score - base_avg_score:+.4f}"
      f"  = {(large_score - base_avg_score) * len(test):+.0f} of {len(test)} test rows")
print("2.8x the parameters and 4x the training time; read the difference against the"
      " fold-to-fold spread before calling it an improvement.")


cardiffnlp/twitter-roberta-large-2022-154m


Loading weights: 100%|██████████| 389/389 [00:00<00:00, 29919.21it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-large-2022-154m
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  fold 0: macro F1 0.6098


Loading weights: 100%|██████████| 389/389 [00:00<00:00, 29903.86it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-large-2022-154m
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  fold 1: macro F1 0.6068


Loading weights: 100%|██████████| 389/389 [00:00<00:00, 40925.68it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-large-2022-154m
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  fold 2: macro F1 0.6340


Loading weights: 100%|██████████| 389/389 [00:00<00:00, 43225.36it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-large-2022-154m
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  fold 3: macro F1 0.6188


Loading weights: 100%|██████████| 389/389 [00:00<00:00, 31100.90it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-large-2022-154m
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  fold 4: macro F1 0.5796
  out-of-fold macro F1 0.6101  (758s)

large  0.6101   base (seed-averaged) 0.6120   difference -0.0018  = -3 of 1440 test rows
2.8x the parameters and 4x the training time; read the difference against the fold-to-fold spread before calling it an improvement.


### 3.5 Calibrating and blending the three models

Three models that score similarly can still be worth combining, because what matters is whether they are wrong about the *same* rows. The linear model reads surface words, the transformers read context, and the blend below is chosen on out-of-fold macro F1 rather than on the leaderboard.

In [13]:
# Blending the three models, and the two things that have to be got right first.
#
# (1) Scale. The linear model returns SVM margins, the transformers return logits, and
#     the two live on different scales. Pushed through softmax as they are, the linear
#     model looks nearly uniform - unconfident - and a weight search then spends its
#     freedom compensating for that rather than deciding which model to trust. A first
#     attempt without this step put 0.85 on the weakest of the three. Fitting one
#     temperature per model on out-of-fold log loss removes the scale difference, after
#     which the weights mean what they look like.
#
# (2) Honesty. Weights chosen on the out-of-fold predictions and then scored on those
#     same predictions are optimistic. The nested loop below picks the weights on four
#     folds and scores them on the fifth, rotating, so the number reported is what this
#     procedure produces on rows it has not seen.
from itertools import product

from scipy.optimize import minimize_scalar
from sklearn.metrics import log_loss

# The linear model again, on the same folds, but keeping decision_function instead of the
# predicted label: a label cannot be averaged with another model's logits, a margin can.
BEST_VIEWS, BEST_C = ["word", "char", "emb"], 0.05
linear_oof = np.zeros((len(train), len(LABELS)), dtype=np.float32)
for fold in range(N_SPLITS):
    is_validation = fold_ids == fold
    X_fit, X_apply = build_features(texts[~is_validation], texts[is_validation],
                                    train_embeddings[~is_validation],
                                    train_embeddings[is_validation], BEST_VIEWS)
    fitted = LinearSVC(C=BEST_C, class_weight="balanced",
                       random_state=RANDOM_STATE, max_iter=10000).fit(X_fit, y_train[~is_validation])
    linear_oof[is_validation] = fitted.decision_function(X_apply)
X_fit, X_apply = build_features(texts, test["text"].values,
                                train_embeddings, test_embeddings, BEST_VIEWS)
linear_test = LinearSVC(C=BEST_C, class_weight="balanced", random_state=RANDOM_STATE,
                        max_iter=10000).fit(X_fit, y_train).decision_function(X_apply)
print(f"linear model, same folds: macro F1 "
      f"{f1_score(y_index, linear_oof.argmax(1), average='macro'):.4f}")

COMPONENTS = {"base_avg": (base_avg_oof, base_avg_test),
              "large":    (large_oof,    large_test),
              "linear":   (linear_oof,   linear_test)}


def softmax(scores, temperature=1.0):
    shifted = scores / temperature
    shifted = shifted - shifted.max(axis=1, keepdims=True)
    exponentiated = np.exp(shifted)
    return exponentiated / exponentiated.sum(axis=1, keepdims=True)


temperatures = {}
print("\ntemperature fitted on out-of-fold log loss")
for name, (oof, _) in COMPONENTS.items():
    fitted = minimize_scalar(
        lambda t: log_loss(y_index, softmax(oof, t), labels=list(range(len(LABELS)))),
        bounds=(0.05, 20.0), method="bounded")
    temperatures[name] = fitted.x
    print(f"  {name:9s} T={fitted.x:5.2f}   log loss {fitted.fun:.4f}"
          f"   own macro F1 {f1_score(y_index, oof.argmax(1), average='macro'):.4f}")

names = list(COMPONENTS)
WEIGHT_GRID = [np.array(list(w) + [1.0 - sum(w)])
               for w in product(np.arange(0, 1.01, 0.05), repeat=len(names) - 1)
               if sum(w) <= 1.0 + 1e-9]


def mix(rows, weights, which):
    """Weighted average of the calibrated probabilities, for the given rows."""
    return sum(weights[i] * softmax(COMPONENTS[n][which][rows], temperatures[n])
               for i, n in enumerate(names))


def best_weights(rows):
    scored = [(f1_score(y_index[rows], mix(rows, w, 0).argmax(1), average="macro"), w)
              for w in WEIGHT_GRID]
    return max(scored, key=lambda pair: pair[0])


everything = np.ones(len(y_index), dtype=bool)
optimistic, chosen_weights = best_weights(everything)
print("\nweights chosen on everything:",
      {n: round(float(chosen_weights[i]), 2) for i, n in enumerate(names)},
      f"-> macro F1 {optimistic:.4f} (optimistic)")

held_out = np.empty(len(y_index), dtype=int)
for fold in range(N_SPLITS):
    _, weights = best_weights(fold_ids != fold)
    held_out[fold_ids == fold] = mix(fold_ids == fold, weights, 0).argmax(1)
honest = f1_score(y_index, held_out, average="macro")
print(f"nested (weights from four folds, scored on the fifth): macro F1 {honest:.4f}")
for name, (oof, _) in COMPONENTS.items():
    own = f1_score(y_index, oof.argmax(1), average="macro")
    print(f"  vs {name:9s} {own:.4f}   gain {honest - own:+.4f}"
          f"   ({(honest - own) * len(test):+.0f} of {len(test)} test rows)")

blend_test = sum(chosen_weights[i] * softmax(COMPONENTS[n][1], temperatures[n])
                 for i, n in enumerate(names))
blend_labels = [LABELS[i] for i in blend_test.argmax(1)]
submission = check_and_write(test["post_id"].values, blend_labels, "blend_calibrated")
print(f"\nFINAL ENTRY: nested out-of-fold macro F1 {honest:.4f}")
print(submission["label"].value_counts().to_string())


linear model, same folds: macro F1 0.5708

temperature fitted on out-of-fold log loss
  base_avg  T= 1.50   log loss 1.0462   own macro F1 0.6120
  large     T= 1.87   log loss 1.0552   own macro F1 0.6101
  linear    T= 0.28   log loss 1.1614   own macro F1 0.5708

weights chosen on everything: {'base_avg': 0.25, 'large': 0.25, 'linear': 0.5} -> macro F1 0.6288 (optimistic)
nested (weights from four folds, scored on the fifth): macro F1 0.6236
  vs base_avg  0.6120   gain +0.0117   (+17 of 1440 test rows)
  vs large     0.6101   gain +0.0135   (+19 of 1440 test rows)
  vs linear    0.5708   gain +0.0529   (+76 of 1440 test rows)
all five checks passed for 'blend_calibrated' (1440 rows, 6 distinct labels)
   written: ..\_workbench\kaggle\submissions\blend_calibrated__id.csv
   written: ..\_workbench\kaggle\submissions\blend_calibrated__post_id.csv

FINAL ENTRY: nested out-of-fold macro F1 0.6236
label
joy         406
anger       324
sadness     255
surprise    181
disgust     148
fear 

### 3.6 Starting from models that already know emotions

Everything in 3.4 started from a model that had read 154 million tweets but had never been told what an emotion is: its classifier head began as random numbers. The checkpoints in this section were already fine-tuned on emotion datasets, and their heads already separate anger, disgust, fear, joy, sadness and surprise, alongside classes this task does not have, such as neutral. Keeping the head rows for our six labels means training starts from a model that can already tell the emotions apart, and our 3,360 tweets only have to adapt it to this corpus.

**GPU kernel again.** The five checkpoints below take about 45 minutes in total on an RTX 5090, most of it the two large models.

In [14]:
# Loading an emotion checkpoint without throwing its head away.
#
# finetune_cv in 3.4 asks for num_labels=6 with ignore_mismatched_sizes=True, which
# discards any head whose size differs from six and starts a new one at random. That
# is right for a checkpoint with no head worth keeping, and wrong here: a 7-class
# emotion head already holds a direction for each of our six emotions. So the head is
# cut down to our six rows instead of replaced. The training loop is otherwise the one
# from 3.4, line for line, so the numbers stay comparable with that section.

# 3.6 and 3.7 reuse what 3.4 and 3.5 computed in this kernel session. If the kernel was
# restarted since then, stop here and run the notebook from the top first.
for needed in ("finetune_cv", "SEEDS", "base_avg_oof", "linear_oof", "honest", "check_and_write"):
    if needed not in globals():
        raise NameError(f"{needed} is not defined: run sections 1 to 3.5 above first")

def load_with_emotion_head(model_name):
    """Load a checkpoint and keep only the classifier rows for our six labels."""
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    # id2label maps each output row of the original head to its label name.
    source_row = {str(name).lower(): int(row) for row, name in model.config.id2label.items()}
    missing = [label for label in LABELS if label not in source_row]
    if missing:
        raise ValueError(f"{model_name} has no output for {missing}")
    rows = [source_row[label] for label in LABELS]          # reordered to match LABELS
    old = model.classifier.out_proj
    new = torch.nn.Linear(old.in_features, len(rows))
    new.weight.data = old.weight.data[rows].clone()         # one weight row per kept label
    new.bias.data = old.bias.data[rows].clone()
    model.classifier.out_proj = new
    model.config.num_labels = model.num_labels = len(rows)
    return model.cuda()


def finetune_emotion_cv(model_name, seed=RANDOM_STATE, epochs=5, lr=3e-5, batch_size=32, max_len=96):
    """finetune_cv from 3.4 with one change: the model keeps its emotion head."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    oof = np.zeros((len(train), len(LABELS)), dtype=np.float32)
    test_logits = np.zeros((len(test), len(LABELS)), dtype=np.float32)
    started = time.time()

    for fold in range(N_SPLITS):
        is_validation = fold_ids == fold
        tokeniser = AutoTokenizer.from_pretrained(model_name)
        model = load_with_emotion_head(model_name)          # the only line that differs

        loader = DataLoader(TweetDataset(texts[~is_validation], y_index[~is_validation],
                                         tokeniser, max_len),
                            batch_size=batch_size, shuffle=True)
        optimiser = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
        schedule = torch.optim.lr_scheduler.OneCycleLR(
            optimiser, max_lr=lr, total_steps=len(loader) * epochs, pct_start=0.1)

        model.train()
        for _ in range(epochs):
            for batch in loader:
                batch = {k: v.cuda() for k, v in batch.items()}
                with torch.autocast("cuda", dtype=torch.bfloat16):
                    logits = model(**{k: v for k, v in batch.items() if k != "labels"}).logits
                    loss = F.cross_entropy(logits, batch["labels"])
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimiser.step()
                schedule.step()
                optimiser.zero_grad(set_to_none=True)

        oof[is_validation] = predict_logits(
            model, DataLoader(TweetDataset(texts[is_validation], None, tokeniser, max_len),
                              batch_size=128))
        test_logits += predict_logits(
            model, DataLoader(TweetDataset(test["text"].values, None, tokeniser, max_len),
                              batch_size=128)) / N_SPLITS
        print(f"  fold {fold}: macro F1 "
              f"{f1_score(y_index[is_validation], oof[is_validation].argmax(1), average='macro'):.4f}",
              flush=True)
        del model
        torch.cuda.empty_cache()

    score = f1_score(y_index, oof.argmax(1), average="macro")
    print(f"  out-of-fold macro F1 {score:.4f}  ({time.time() - started:.0f}s)", flush=True)
    return oof, test_logits, score


In [15]:
# Two checkpoints from the same author, three seeds each, averaged the way the base
# model was in 3.4. Both were fine-tuned on six English emotion datasets with seven
# classes, our six plus neutral; one is distilled (82 million parameters), the other
# large (355 million). The large one takes the smaller learning rate and batch that
# the large model in 3.4 needed.
EMOTION_RUNS = {
    "emo_distil": ("j-hartmann/emotion-english-distilroberta-base", dict(lr=3e-5, batch_size=32)),
    "emo_large":  ("j-hartmann/emotion-english-roberta-large",      dict(lr=1.5e-5, batch_size=16)),
}

emotion_oof, emotion_test = {}, {}
for tag, (checkpoint, settings) in EMOTION_RUNS.items():
    runs = []
    for seed in SEEDS:
        print(f"{checkpoint}  seed={seed}")
        runs.append(finetune_emotion_cv(checkpoint, seed=seed, **settings))
    # Keep the first seed on its own as well: the four-model blend in 3.7 uses it.
    emotion_oof[tag + "_s42"], emotion_test[tag + "_s42"] = runs[0][0], runs[0][1]
    emotion_oof[tag + "_avg"] = np.mean([r[0] for r in runs], axis=0)
    emotion_test[tag + "_avg"] = np.mean([r[1] for r in runs], axis=0)
    seed_scores = [r[2] for r in runs]
    averaged = f1_score(y_index, emotion_oof[tag + "_avg"].argmax(1), average="macro")
    print(f"\n{tag}: seeds {[round(s, 4) for s in seed_scores]}"
          f" (spread {max(seed_scores) - min(seed_scores):.4f}), averaged {averaged:.4f}\n")

print(f"tweet-domain base, seed-averaged (3.4): {base_avg_score:.4f}")
for tag in EMOTION_RUNS:
    averaged = f1_score(y_index, emotion_oof[tag + "_avg"].argmax(1), average="macro")
    print(f"{tag:10s} seed-averaged         : {averaged:.4f}   ({averaged - base_avg_score:+.4f})")


j-hartmann/emotion-english-distilroberta-base  seed=42


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 26258.16it/s]


  fold 0: macro F1 0.6252


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 26250.34it/s]


  fold 1: macro F1 0.6179


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 35033.17it/s]


... (51 outputs left out)

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 38625.93it/s]


  fold 3: macro F1 0.6477


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 26023.20it/s]


  fold 4: macro F1 0.5862
  out-of-fold macro F1 0.6214  (492s)

emo_large: seeds [0.6173, 0.6253, 0.6214] (spread 0.0080), averaged 0.6242

tweet-domain base, seed-averaged (3.4): 0.6120
emo_distil seed-averaged         : 0.6262   (+0.0143)
emo_large  seed-averaged         : 0.6242   (+0.0122)


In [16]:
# Three more emotion checkpoints, one seed each. They were trained on different text,
# which is the point: a blend gains only where its members are wrong about different rows.
#   go_emotions   - Reddit comments, 28 emotion classes
#   michellejieli - dialogue from the TV series Friends, 7 classes
#   cardiffnlp    - tweets again, 11 emotion classes, large model
MORE_EMOTION = {
    "emo_goemo":     ("SamLowe/roberta-base-go_emotions",                 dict(lr=3e-5, batch_size=32)),
    "emo_michelle":  ("michellejieli/emotion_text_classifier",            dict(lr=3e-5, batch_size=32)),
    "emo_cardlarge": ("cardiffnlp/twitter-roberta-large-emotion-latest",  dict(lr=1.5e-5, batch_size=16)),
}
for tag, (checkpoint, settings) in MORE_EMOTION.items():
    print(checkpoint)
    emotion_oof[tag], emotion_test[tag], score = finetune_emotion_cv(checkpoint, **settings)
    print(f"{tag}: {score:.4f}\n")


SamLowe/roberta-base-go_emotions


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 880.20it/s]


  fold 0: macro F1 0.6416


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1112.89it/s]


  fold 1: macro F1 0.6203


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1201.67it/s]


... (21 outputs left out)

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 509.73it/s]


  fold 3: macro F1 0.6192


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 647.43it/s]


  fold 4: macro F1 0.5807
  out-of-fold macro F1 0.6059  (676s)
emo_cardlarge: 0.6059



### 3.7 The final blend

The same calibrated blend as 3.5, rebuilt with the emotion models. Two versions are written. The four-model one mirrors the entry that scored best on the public leaderboard; the seven-model one has the best nested out-of-fold score. The final two submissions are chosen by out-of-fold score, not by the 432-row public leaderboard.

In [17]:
# The blend from 3.5, wrapped in a function so it can be run on any set of models.
# Same three steps: one temperature per model fitted on out-of-fold log loss, a weight
# grid searched on macro F1, and a nested estimate (weights from four folds, scored on
# the fifth). The grid step is 0.05 for four models and 0.1 for seven, because the
# number of weight combinations grows quickly with the number of models.

def calibrated_blend(components, step):
    names = list(components)
    temps = {}
    for n in names:
        oof_scores = components[n][0]
        temps[n] = minimize_scalar(
            lambda t: log_loss(y_index, softmax(oof_scores, t), labels=list(range(len(LABELS)))),
            bounds=(0.05, 20.0), method="bounded").x
    p_oof = [softmax(components[n][0], temps[n]) for n in names]
    p_test = [softmax(components[n][1], temps[n]) for n in names]
    grid = [np.array(list(w) + [1.0 - sum(w)])
            for w in product(np.arange(0, 1.0 + 1e-9, step), repeat=len(names) - 1)
            if sum(w) <= 1.0 + 1e-9]

    def pick(rows):
        scored = [(f1_score(y_index[rows], sum(w[i] * p_oof[i][rows] for i in range(len(names))).argmax(1),
                            average="macro"), w) for w in grid]
        return max(scored, key=lambda pair: pair[0])

    optimistic, weights = pick(np.ones(len(y_index), dtype=bool))
    held = np.empty(len(y_index), dtype=int)
    per_fold = []
    for fold in range(N_SPLITS):
        _, w = pick(fold_ids != fold)
        rows = fold_ids == fold
        held[rows] = sum(w[i] * p_oof[i][rows] for i in range(len(names))).argmax(1)
        per_fold.append(f1_score(y_index[rows], held[rows], average="macro"))
    nested = f1_score(y_index, held, average="macro")

    print(f"{len(names)} models, grid step {step}")
    for i, n in enumerate(names):
        own = f1_score(y_index, components[n][0].argmax(1), average="macro")
        print(f"  {n:14s} own {own:.4f}   T={temps[n]:5.2f}   weight {weights[i]:.2f}")
    print(f"  optimistic {optimistic:.4f} | nested {nested:.4f} | per fold {np.round(per_fold, 4)}")
    test_probs = sum(weights[i] * p_test[i] for i in range(len(names)))
    return nested, per_fold, test_probs


FOUR = {"base_avg":   (base_avg_oof, base_avg_test),
        "emo_large":  (emotion_oof["emo_large_s42"], emotion_test["emo_large_s42"]),
        "emo_distil": (emotion_oof["emo_distil_s42"], emotion_test["emo_distil_s42"]),
        "linear":     (linear_oof, linear_test)}
SEVEN = {"base_avg":       (base_avg_oof, base_avg_test),
         "emo_large_avg":  (emotion_oof["emo_large_avg"], emotion_test["emo_large_avg"]),
         "emo_distil_avg": (emotion_oof["emo_distil_avg"], emotion_test["emo_distil_avg"]),
         "linear":         (linear_oof, linear_test),
         "emo_goemo":      (emotion_oof["emo_goemo"], emotion_test["emo_goemo"]),
         "emo_michelle":   (emotion_oof["emo_michelle"], emotion_test["emo_michelle"]),
         "emo_cardlarge":  (emotion_oof["emo_cardlarge"], emotion_test["emo_cardlarge"])}

print(f"3.5 blend without emotion models, nested: {honest:.4f}\n")
four_nested, four_folds, four_probs = calibrated_blend(FOUR, 0.05)
print()
seven_nested, seven_folds, seven_probs = calibrated_blend(SEVEN, 0.1)

print(f"\nfour models  vs 3.5: {four_nested - honest:+.4f}  = {(four_nested - honest) * len(test):+.0f} of {len(test)} test rows")
print(f"seven models vs four: {seven_nested - four_nested:+.4f}  = {(seven_nested - four_nested) * len(test):+.0f} rows,"
      f" up in {sum(s > f for s, f in zip(seven_folds, four_folds))} of {N_SPLITS} folds")

print()
for stem, probs in (("final_emo4", four_probs), ("final_emo7", seven_probs)):
    labels_out = [LABELS[i] for i in probs.argmax(1)]
    written = check_and_write(test["post_id"].values, labels_out, stem)
    print("   ", written["label"].value_counts().to_dict())


3.5 blend without emotion models, nested: 0.6236

4 models, grid step 0.05
  base_avg       own 0.6120   T= 1.50   weight 0.25
  emo_large      own 0.6173   T= 2.03   weight 0.15
  emo_distil     own 0.6169   T= 1.72   weight 0.35
  linear         own 0.5708   T= 0.28   weight 0.25
  optimistic 0.6470 | nested 0.6438 | per fold [0.6492 0.6337 0.6562 0.6579 0.62  ]

7 models, grid step 0.1
  base_avg       own 0.6120   T= 1.50   weight 0.10
  emo_large_avg  own 0.6242   T= 1.98   weight 0.20
  emo_distil_avg own 0.6262   T= 1.70   weight 0.20
  linear         own 0.5708   T= 0.28   weight 0.10
  emo_goemo      own 0.6222   T= 1.78   weight 0.20
  emo_michelle   own 0.6014   T= 1.73   weight 0.00
  emo_cardlarge  own 0.6059   T= 2.49   weight 0.20
  optimistic 0.6555 | nested 0.6536 | per fold [0.6713 0.6403 0.652  0.672  0.63  ]

four models  vs 3.5: +0.0201  = +29 of 1440 test rows
seven models vs four: +0.0098  = +14 rows, up in 4 of 5 folds

all five checks passed for 'final_emo4' (1